# STOP/APO analysis — MS, C, D and E

**Clean working version based on `C_el_STOP_APO(3).ipynb`.**

This notebook consolidates the four lineage sections. It keeps the biological categories separate: `DIVIDE`, `STOP`, `APO`, and virtual `NULL` positions. For the binary STOP-vs-DIVIDE comparison, STOP is coded as 1 and DIVIDE as 0; APO is not silently recoded as STOP. A second comparison handles APO vs non-APO.

The established lineage counts are used as validation targets: MS = 14 APO across 16 subtrees; C = 1 APO; D = 0 APO; E = 0 APO. No missing lineage state is inferred from absence of descendants.

**Input files** (upload together in the first cell): `TR_CATALOG_RAW.tsv`, `MS_BOTTOM_UP_TOP_DOWN.txt`, `C_BOTTOM_UP_TOP_DOWN.txt`, `D_BOTTOM_UP_TOP_DOWN.txt`, `E_BOTTOM_UP_TOP_DOWN.txt`. The TR catalogue must be the validated `TR_CATALOG_RAW.tsv`, not a derived/final catalogue.


In [ ]:
# 1. Upload input files (Google Colab)
from google.colab import files
from pathlib import Path

uploaded = files.upload()
BASE = Path('/content')
required = [
    'TR_CATALOG_RAW.tsv',
    'MS_BOTTOM_UP_TOP_DOWN.txt',
    'C_BOTTOM_UP_TOP_DOWN.txt',
    'D_BOTTOM_UP_TOP_DOWN.txt',
    'E_BOTTOM_UP_TOP_DOWN.txt',
]
missing = [name for name in required if not (BASE / name).exists()]
if missing:
    raise FileNotFoundError('Missing required input files: ' + ', '.join(missing))
print('All required input files are available.')


In [ ]:
# 2. Imports and validated TR catalogue
import re
import math
import json
import numpy as np
import pandas as pd
from collections import deque

TR_RAW = pd.read_csv(BASE / 'TR_CATALOG_RAW.tsv', sep='\t', low_memory=False)
required_columns = {'Chromosome','Start','End','Period','Repeat_Sequence'}
missing_columns = required_columns - set(TR_RAW.columns)
if missing_columns:
    raise ValueError(f'TR_CATALOG_RAW.tsv is missing columns: {sorted(missing_columns)}')
TR_RAW['Period'] = pd.to_numeric(TR_RAW['Period'], errors='coerce')
TR_RAW['Repeat_Sequence'] = TR_RAW['Repeat_Sequence'].astype(str).str.upper()
TR_RAW = TR_RAW[TR_RAW['Period'].notna() & (TR_RAW['Period'] > 0)].copy()
TR_RAW['Period'] = TR_RAW['Period'].astype(int)
TR_RAW['Repeat_Length'] = TR_RAW['Repeat_Sequence'].str.len()
TR_RAW['N_monomeri'] = TR_RAW['Repeat_Length'] // TR_RAW['Period']
TR_RAW['Coda_esclusa'] = TR_RAW['Repeat_Length'] % TR_RAW['Period']
TR_RAW['Sequenza_completa'] = [
    seq[:(len(seq)//periodo)*periodo]
    for seq, periodo in zip(TR_RAW['Repeat_Sequence'], TR_RAW['Period'])
]
print(f'Validated raw TR rows: {len(TR_RAW):,}')
print('TRs with at least one complete monomer:', int((TR_RAW['N_monomeri'] >= 1).sum()))


## 3. Read and validate lineage source files

The parser reads the named STOP, DIVIDE and APO sections and the complete biological tree. Section totals are checked against the known lineage totals. It does not assign a state to an unclassified node.

In [ ]:
# 3. Lineage parser
SECTION_MARKERS = {
    'MS': ['========== STOP (80) ==========', '========== DIVIDE (93) ==========', '========== APO (14) ==========' ],
    'C':  ['========== STOP (47) ==========', '========== DIVIDE (47) ==========', '========== APO (1) ==========' ],
    'D':  ['========== STOP (20) ==========', '========== DIVIDE (19) ==========', '========== APO (0) ==========' ],
    'E':  ['========== STOP (20) ==========', '========== DIVIDE (19) ==========', '========== APO (0) ==========' ],
}
FILES = {k: f'{k}_BOTTOM_UP_TOP_DOWN.txt' for k in SECTION_MARKERS}

def _section(lines, title, all_markers):
    if title not in lines:
        raise ValueError(f'Section marker not found: {title}')
    start = lines.index(title) + 1
    later = [lines.index(m) for m in all_markers if m in lines and lines.index(m) >= start]
    end = min(later) if later else len(lines)
    return [x.strip() for x in lines[start:end] if x.strip()]

def read_lineage(name):
    path = BASE / FILES[name]
    lines = [x.strip() for x in path.read_text(encoding='utf-8-sig').splitlines()]
    markers = SECTION_MARKERS[name]
    tree_marker = '-- ALBERO COMPLETO --'
    if tree_marker not in lines:
        raise ValueError(f'{name}: complete-tree marker not found')
    tree_end = lines.index(markers[0])
    tree_lines = lines[lines.index(tree_marker)+1:tree_end]
    tree = set()
    for line in tree_lines:
        if line == name:
            tree.add('')
        elif line.startswith(name + ' '):
            tree.add(line[len(name)+1:].strip())
    data = {}
    labels = ['STOP','DIVIDE','APO']
    for label, marker in zip(labels, markers):
        rows = _section(lines, marker, markers)
        names = []
        for row in rows:
            if row == name:
                names.append('')
            elif row.startswith(name + ' '):
                names.append(row[len(name)+1:].strip())
        data[label] = set(names)
    if not tree:
        raise ValueError(f'{name}: no biological nodes parsed')
    assigned = data['STOP'] | data['DIVIDE'] | data['APO']
    duplicates = sum(len(data[k]) for k in data) - len(assigned)
    if duplicates:
        raise ValueError(f'{name}: a node occurs in more than one state section')
    unclassified = tree - assigned
    outside = assigned - tree
    if unclassified or outside:
        raise ValueError(f'{name}: unclassified tree nodes={len(unclassified)}; state nodes outside tree={len(outside)}')
    return {'tree': tree, **data}

LINEAGES = {name: read_lineage(name) for name in FILES}
EXPECTED = {'MS': (80,93,14), 'C': (47,47,1), 'D': (20,19,0), 'E': (20,19,0)}
for name, data in LINEAGES.items():
    observed = tuple(len(data[k]) for k in ('STOP','DIVIDE','APO'))
    if observed != EXPECTED[name]:
        raise ValueError(f'{name}: expected STOP/DIVIDE/APO={EXPECTED[name]}, observed={observed}')
    print(f'{name}: tree={len(data["tree"])} | STOP={observed[0]} | DIVIDE={observed[1]} | APO={observed[2]}')


## 4. Normalize lineage positions

MS is partitioned into its 16 established four-letter subtrees. C is partitioned into its eight three-letter roots. D and E are each kept as a single lineage tree. The original branch order is retained where the lineage nomenclature provides it. Internal virtual positions are represented as `NULL`; they are never treated as 0 or 1.

In [ ]:
# 4. Normalization helpers
MS_ROOTS = ['aaaa','aaap','aapa','aapp','apaa','apap','appa','appp',
            'paaa','paap','papa','papp','ppaa','ppap','pppa','pppp']
C_ROOTS = ['aaa','aap','apa','app','paa','pap','ppa','ppp']
C_CHILD_ORDER = ('a','p','l','r','d','v')

def state_of(data, node):
    for state in ('DIVIDE','STOP','APO'):
        if node in data[state]:
            return state
    raise ValueError(f'Node has no explicit state: {node!r}')

def normalize_ms(data):
    rows=[]
    for root in MS_ROOTS:
        nodes=sorted((n for n in data['tree'] if n.startswith(root)), key=lambda n:(len(n),n))
        if not nodes: raise ValueError(f'MS subtree missing: {root}')
        max_depth=max(len(n) for n in nodes)
        positions=[]
        for node in nodes:
            state=state_of(data,node)
            positions.append({'Radice':root,'Nodo':node,'Stato':state,'Codice': '0' if state=='DIVIDE' else '1' if state=='STOP' else 'APO','Tipo_posizione':'BIOLOGICA'})
            if state=='APO' or (state=='STOP' and len(node)<max_depth):
                for j in (1,2): positions.append({'Radice':root,'Nodo':f'{node}_NULL{j}','Stato':'NULL','Codice':'NULL','Tipo_posizione':'VIRTUALE'})
        for i,row in enumerate(positions,1): row['Posizione_finale']=i; rows.append(row)
    return pd.DataFrame(rows)

def normalize_binary_tree(name, data, roots, child_order):
    rows=[]
    for root in roots:
        nodes={n for n in data['tree'] if n.startswith(root)}
        if root not in nodes: raise ValueError(f'{name}: expected subtree root not found: {root}')
        max_depth=max(len(n)-len(root) for n in nodes)
        queue=deque([(root,0,0)])
        positions=[]
        while queue:
            node, depth, index=queue.popleft()
            if node is None:
                positions.append({'Radice':root,'Nodo':'NULL','Stato':'NULL','Codice':'NULL','Tipo_posizione':'VIRTUALE','Profondita':depth,'Posizione':index})
                if depth<max_depth:
                    queue.extend([(None,depth+1,2*index+1),(None,depth+1,2*index+2)])
                continue
            state=state_of(data,node)
            positions.append({'Radice':root,'Nodo':node,'Stato':state,'Codice':'0' if state=='DIVIDE' else '1' if state=='STOP' else 'APO','Tipo_posizione':'BIOLOGICA','Profondita':depth,'Posizione':index})
            if depth<max_depth:
                if state=='DIVIDE':
                    children=[node+s for s in child_order if node+s in nodes]
                    if len(children)!=2: raise ValueError(f'{name}: {node!r} has {len(children)} children: {children}')
                    queue.extend([(children[0],depth+1,2*index+1),(children[1],depth+1,2*index+2)])
                else:
                    queue.extend([(None,depth+1,2*index+1),(None,depth+1,2*index+2)])
        positions.sort(key=lambda x:x['Posizione'])
        while positions and positions[-1]['Stato']=='NULL': positions.pop()
        for i,row in enumerate(positions,1): row['Posizione_finale']=i; rows.append(row)
    return pd.DataFrame(rows)

NORMALIZED = {
    'MS': normalize_ms(LINEAGES['MS']),
    'C': normalize_binary_tree('C', LINEAGES['C'], C_ROOTS, C_CHILD_ORDER),
    'D': normalize_binary_tree('D', LINEAGES['D'], [''], tuple('abcdefghijklmnopqrstuvwxyz')) ,
    'E': normalize_binary_tree('E', LINEAGES['E'], [''], tuple('abcdefghijklmnopqrstuvwxyz')),
}


## 5. Structural summaries and independent fate categories

The summary reports STOP, DIVIDE, APO and NULL independently. In particular, APO is not merged into STOP in the exported state table.

In [ ]:
# 5. Summaries and consistency checks
summary=[]
for name, df in NORMALIZED.items():
    bio=df[df['Tipo_posizione']=='BIOLOGICA']
    summary.append({'Lineage':name,'Subtrees':bio['Radice'].nunique(),'Biological_nodes':len(bio),
                    'DIVIDE':int((bio.Stato=='DIVIDE').sum()),'STOP':int((bio.Stato=='STOP').sum()),
                    'APO':int((bio.Stato=='APO').sum()),'Internal_NULL':int((df.Stato=='NULL').sum()),
                    'Normalized_positions':len(df)})
SUMMARY=pd.DataFrame(summary)
EXPECTED_APO={'MS':14,'C':1,'D':0,'E':0}
for name, expected in EXPECTED_APO.items():
    observed=int((NORMALIZED[name]['Stato']=='APO').sum())
    if observed!=expected: raise ValueError(f'{name}: expected {expected} APO, found {observed}')
print('Validated APO counts:', EXPECTED_APO)
display(SUMMARY)


## 6. TR candidate preparation

The raw catalogue is retained as the sole TR source. For each lineage/subtree, the scan uses only complete monomers and the actual normalized positions of the relevant state classes. No fixed AB-derived minimum length is imported.

In [ ]:
# 6. Prepare complete monomer matrices from TR_CATALOG_RAW
def tr_matrix(row):
    period=int(row['Period'])
    seq=str(row['Sequenza_completa'])
    if len(seq)%period: raise ValueError('Incomplete monomer sequence after trimming')
    return np.frombuffer(seq.encode('ascii'), dtype=np.uint8).reshape(-1,period)

TR_CATALOG = TR_RAW.copy().reset_index(drop=True)
TR_CATALOG['TR_ID']=[f"TR_{r.Chromosome}_{r.Start}_{r.End}" for r in TR_CATALOG.itertuples()]
print('TR catalogue rows available for scanning:',len(TR_CATALOG))


## 7. Independent scans: STOP vs DIVIDE, and APO vs non-APO

A window is promoted only when every cross-class monomer comparison has identity **below** the threshold. `NULL` positions are excluded. The two scans are independent: the STOP/DIVIDE scan does not include APO; the APO/non-APO scan compares APO against all other biological nodes.

In [ ]:
# 7. Scan functions
IDENTITY_THRESHOLD=0.50

def scan_pairwise(lineage, df_norm, class_a, class_b, threshold=IDENTITY_THRESHOLD):
    results=[]
    for root, group in df_norm.groupby('Radice', sort=True):
        ordered=group.sort_values('Posizione_finale').reset_index(drop=True)
        # Keep virtual NULL slots in the coordinate system; exclude them only from class membership.
        pos_a=[i for i,row in ordered.iterrows() if row['Tipo_posizione']=='BIOLOGICA' and row['Stato']==class_a]
        pos_b=[i for i,row in ordered.iterrows() if row['Tipo_posizione']=='BIOLOGICA' and row['Stato']==class_b]
        if not pos_a or not pos_b: continue
        required=max(pos_a+pos_b)+1
        for _,tr in TR_CATALOG.iterrows():
            n=int(tr['N_monomeri'])
            if n<required: continue
            matrix=tr_matrix(tr)
            for offset in range(n-required+1):
                window=matrix[offset:offset+required]
                passed=True
                for ia in pos_a:
                    for ib in pos_b:
                        identity=float(np.count_nonzero(window[ia]==window[ib])/int(tr['Period']))
                        if identity>=threshold:
                            passed=False; break
                    if not passed: break
                if passed:
                    results.append({'Lineage':lineage,'Radice':root,'Comparison':f'{class_a}_vs_{class_b}',
                                    'TR_ID':tr['TR_ID'],'TR_Indice_RAW':int(tr.name),'Chromosome':tr['Chromosome'],
                                    'Start':tr['Start'],'End':tr['End'],'Period':int(tr['Period']),
                                    'N_monomeri':n,'Window_length':required,'Offset':offset})
    return pd.DataFrame(results)

RESULTS_STOP_DIVIDE=[]
RESULTS_APO=[]
for lineage, df in NORMALIZED.items():
    print(f'Running {lineage}...')
    RESULTS_STOP_DIVIDE.append(scan_pairwise(lineage,df,'STOP','DIVIDE'))
    # APO is compared with every other biological position; if APO is absent, return an empty result.
    if (df['Stato']=='APO').any():
        temp=df.copy()
        temp['Stato_APO_SCAN']=np.where(temp['Stato']=='APO','APO',np.where(temp['Tipo_posizione']=='BIOLOGICA','NON_APO','NULL'))
        # Scan helper uses the original biological state column; make a scan-specific copy.
        temp.loc[temp['Tipo_posizione']=='BIOLOGICA','Stato']=temp.loc[temp['Tipo_posizione']=='BIOLOGICA','Stato_APO_SCAN']
        RESULTS_APO.append(scan_pairwise(lineage,temp,'APO','NON_APO'))
    else:
        RESULTS_APO.append(pd.DataFrame())

DF_STOP_DIVIDE=pd.concat(RESULTS_STOP_DIVIDE,ignore_index=True)
DF_APO=pd.concat(RESULTS_APO,ignore_index=True)
print('STOP vs DIVIDE windows:',len(DF_STOP_DIVIDE))
print('APO vs non-APO windows:',len(DF_APO))


## 8. Export

Exports include normalized lineage positions, the four-lineage summary, and the two independent TR scan result tables. These are new outputs; source files are not modified.

In [ ]:
# 8. Save outputs
OUT=BASE/'STOP_APO_MS_C_D_E_results'
OUT.mkdir(exist_ok=True)
SUMMARY.to_csv(OUT/'LINEAGE_SUMMARY.csv',index=False)
for lineage,df in NORMALIZED.items():
    df.to_csv(OUT/f'{lineage}_NORMALIZED_POSITIONS.csv',index=False)
DF_STOP_DIVIDE.to_csv(OUT/'TR_STOP_vs_DIVIDE.csv',index=False)
DF_APO.to_csv(OUT/'TR_APO_vs_NON_APO.csv',index=False)
print('Saved outputs to:',OUT)
